In [ ]:
import requests
import pandas as pd
import pickle
from time import sleep
from random import shuffle

In [ ]:
# df = pd.read_pickle("./listaPlayersNomeJogo.pkl")
df = pd.read_pickle("./jogosInfo.pkl")
df

,num,nome,currentPlayers,peak24h,allTimePeak,gameID,dados,tags
0,1,Counter-Strike 2,1274295,1527573,1818773,730,"{'type': 'game', 'name': 'Counter-Strike 2', '...","[FPS, Shooter, Multiplayer, Competitive, Actio..."
1,2,Dota 2,697691,716906,1295114,570,"{'type': 'game', 'name': 'Dota 2', 'steam_appi...","[Free to Play, MOBA, Multiplayer, Strategy, eS..."
2,3,PUBG: BATTLEGROUNDS,450881,631467,3257248,578080,"{'type': 'game', 'name': 'PUBG: BATTLEGROUNDS'...","[Survival, Shooter, Battle Royale, Multiplayer..."
3,4,Apex Legends,245878,357225,624473,1172470,"{'type': 'game', 'name': 'Apex Legends™', 'ste...","[Free to Play, Battle Royale, Multiplayer, FPS..."
4,5,Source SDK Base 2007,168144,187294,221857,218,None,None
...,...,...,...,...,...,...,...,...
4995,4996,Devil Slayer - Raksasi / 斩妖Raksasi,20,26,1080,1016600,"{'type': 'game', 'name': 'Devil Slayer - Raksa...","[Difficult, Action Roguelike, Top-Down, Action..."
4996,4997,DEEEER Simulator: Your Average Everyday Deer Game,20,25,251,1018800,"{'type': 'game', 'name': 'DEEEER Simulator: Yo...","[Exploration, Sandbox, Physics, Funny, Third P..."
4997,4998,Wanba Warriors,20,31,720,1021770,"{'type': 'game', 'name': ' Wanba Warriors', 's...","[Fighting, Multiplayer, Funny, Parody , PvP, L..."
4998,4999,Coloring Game,20,31,593,1026820,"{'type': 'game', 'name': 'Coloring Game', 'ste...","[Free to Play, Relaxing, Pixel Graphics, Casua..."


In [ ]:
df.iloc[0]["dados"]

{'type': 'game',
 'name': 'Counter-Strike 2',
 'steam_appid': 730,
 'required_age': 0,
 'is_free': True,
 'dlc': [2678630],
 'detailed_description': 'For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.<br><br>A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools.<br><br>In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features:<br><br><ul class="bb_ul"><li>All-new CS Ratings with the updated Premier mode<br></li><li>Global and Regional leaderboards<br></li><li>Upgraded and overhauled maps<br></li><li>Game-changing dynamic smoke grenades<br

In [ ]:
# df["dados"] = None

In [ ]:
lastFail = None

In [ ]:
# Realiza uma solicitação GET à API da Steam para obter detalhes de um aplicativo específico e converte a resposta JSON em um dicionário
requests.get(f"https://store.steampowered.com/api/appdetails?appids={lastFail}&cc=US").json()

{'2088380': {'success': False}}

In [ ]:
if lastFail is not None:

    tempo = 1
    sucesso = False
    # Loop para tentar obter informações até ter sucesso
    while not sucesso:
        try:
            getInfo(lastFail)
            sucesso = True
        except:
            print(f"Sleeping {tempo}")
            sleep(tempo)
            tempo = 2*tempo


Sleeping 1
Sleeping 2
Sleeping 4
Sleeping 8
Sleeping 16
Sleeping 32
Sleeping 64
Sleeping 128


In [ ]:
# Conjunto para armazenar os índices das linhas onde a coluna 'dados' é None
indicesNones = set()
for i in range(len(df)):
    if df.iloc[i]["dados"] is None:
        indicesNones.add(i)

len(indicesNones)

172

In [ ]:
# Função para obter informações do aplicativo a partir da API da Steam
def getInfo(appId):
    global lastFail
    try:
        aux = requests.get(f"https://store.steampowered.com/api/appdetails?appids={appId}&cc=US").json()
        if aux is None:
            lastFail = appId
            raise Exception(f"Parou de retornar com id={appId}")
            return None
        if aux[f"{appId}"]["success"] == False:
            return None
        data = aux[f"{appId}"]["data"]
    except Exception as e:
        # print(appId)
        raise e
    return data


# Tenta novamente obter informações do último aplicativo falhado, se existir
if lastFail is not None :
    try:
        getInfo(lastFail)
    except Exception as e:
        raise e

# Inicializa o contador de modificados
modificados = 0
# Converte o conjunto de índices None para uma lista e a embaralha
lista = list(indicesNones)
shuffle(lista)
for indice in lista:
    # Se a coluna 'dados' não for None, remove o índice do conjunto
    if df.iloc[indice]["dados"] is not None:
        indicesNones.remove(indice)
        continue

    # Obtém o ID do jogo
    id = df.iloc[indice]["gameID"]

    # Obtém as informações do jogo usando a função getInfo
    data = getInfo(id)

    # Se os dados não forem None, atualiza o DataFrame e incrementa o contador de modificados
    if data is not None:
        indicesNones.remove(indice)
        df.at[indice, "dados"] = data
        modificados += 1



print(f"Modificados: {modificados}")

Modificados: 0


In [ ]:
# Calcula a soma dos valores na coluna 'valor' para as linhas onde a coluna 'dados' não é None e onde é None
aux1 = sum([linha[2] for linha in df.values if linha[6] is not None])
aux2 = sum([linha[2] for linha in df.values if linha[6] is None])
print(f"Valores não nulos: {aux1}")
print(f"Valores nulos: {aux2}")
print(f"{(100 * aux2/(aux1+aux2)):.2f}%")

Valores não nulos: 8446899
Valores nulos: 253054
2.91%


In [ ]:
# Configura o pandas para exibir até 5000 linhas
pd.set_option('display.max_rows', 5000)
# Cria uma máscara para filtrar as linhas onde a coluna 'dados' é None
mask = [(linha[6] is None) for linha in df.values]
df

,num,nome,currentPlayers,peak24h,allTimePeak,gameID,dados
0,1,Counter-Strike 2,1274295,1527573,1818773,730,"{'type': 'game', 'name': 'Counter-Strike 2', '..."
1,2,Dota 2,697691,716906,1295114,570,"{'type': 'game', 'name': 'Dota 2', 'steam_appi..."
2,3,PUBG: BATTLEGROUNDS,450881,631467,3257248,578080,"{'type': 'game', 'name': 'PUBG: BATTLEGROUNDS'..."
3,4,Apex Legends,245878,357225,624473,1172470,"{'type': 'game', 'name': 'Apex Legends™', 'ste..."
4,5,Source SDK Base 2007,168144,187294,221857,218,None
5,6,NARAKA: BLADEPOINT,155758,278864,372076,1203220,"{'type': 'game', 'name': 'NARAKA: BLADEPOINT',..."
6,7,Grand Theft Auto V,127112,136972,364548,271590,"{'type': 'game', 'name': 'Grand Theft Auto V',..."
7,8,Rust,112892,112892,245243,252490,"{'type': 'game', 'name': 'Rust', 'steam_appid'..."
8,9,Stardew Valley,99342,112757,236614,413150,"{'type': 'game', 'name': 'Stardew Valley', 'st..."
9,10,Wallpaper Engine,90378,104463,150375,431960,"{'type': 'game', 'name': 'Wallpaper Engine', '..."


In [ ]:
df.to_pickle("./jogosInfo.pkl")